# Section 2: Preprocessing and Train/Test Split
*Owner: Execution lead*

**Goal:** turn the raw table into clean, numeric, scaled data and split it for training and testing.

**This section hands over these variables** (the next sections depend on the exact names):
`X_train`, `X_test`, `y_train`, `y_test`, `scaler`

Steps: fill missing values, encode text as numbers, check correlation, select features, split, normalize.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

os.makedirs("outputs", exist_ok=True)

df = pd.read_csv("data/Titanic-Dataset.csv")
print("Original shape:", df.shape)

## 2.1 Handle missing values
- **Age:** fill with the **median**. The median is not pulled around by very old or very young outliers.
- **Embarked:** fill with the **mode** (the most common port), because it is a category.
- **Cabin:** most values are missing, so we will drop it later instead of filling it.

In [ ]:
print("Missing values BEFORE:")
print(df[["Age", "Embarked", "Cabin"]].isnull().sum())

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("\nMissing values AFTER (Cabin left for now):")
print(df[["Age", "Embarked", "Cabin"]].isnull().sum())

## 2.2 Encode text columns as numbers
Models only understand numbers.
- **Sex:** male = 0, female = 1 (only two values, so a simple map works).
- **Embarked:** three ports with no natural order, so we use **one-hot encoding** (`get_dummies`): a separate 0/1 column per port. `drop_first=True` removes one column because it can be worked out from the others.

In [ ]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True, dtype=int)

df[["Sex", "Embarked_Q", "Embarked_S"]].head()

## 2.3 Correlation
Correlation (from -1 to +1) shows how strongly two columns move together. We look at the **Survived** row to see which columns relate to survival. Name, Ticket and Cabin are text, so `select_dtypes` leaves them out automatically.

In [ ]:
numeric_df = df.select_dtypes(include="number")

plt.figure(figsize=(9, 7))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.savefig("outputs/correlation_heatmap.png", bbox_inches="tight")
plt.show()

numeric_df.corr()["Survived"].sort_values()

**Observation:** *(fill in from the numbers above: which columns correlate most with Survived? Sex and Pclass are usually the strongest. Notice PassengerId has almost no correlation.)*

## 2.4 Feature selection
We keep columns that carry useful information and drop the rest:

| Dropped | Reason |
|---|---|
| PassengerId | Just a row number, no relation to survival |
| Name | Unique text per person, not usable by a simple model |
| Ticket | Mostly unique codes |
| Cabin | Most values missing |

**Kept:** Pclass, Sex, Age, SibSp, Parch, Fare, Embarked_Q, Embarked_S.

In [ ]:
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

print("Shape after selection:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())
df.head()

## 2.5 Split inputs and answer, then train/test split
- `X` = input columns, `y` = the answer (`Survived`).
- 80% of passengers for training, 20% kept for testing.
- `random_state=42`: the same split every time, so everyone gets the same results.
- `stratify=y`: keeps the survived/not-survived ratio equal in train and test.

In [ ]:
X = df.drop(columns="Survived")
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training passengers:", len(X_train))
print("Testing passengers: ", len(X_test))
print("Survival rate in train:", round(y_train.mean(), 3))
print("Survival rate in test: ", round(y_test.mean(), 3))

## 2.6 Normalization (scaling)
Columns have very different ranges (Fare goes up to hundreds, Sex is 0 or 1). `StandardScaler` rescales each column to mean 0 and standard deviation 1 so no column dominates just because its numbers are bigger.

**Important:** the scaler learns (`fit`) from the **training data only**, then is applied (`transform`) to both sets. If it learned from the test data, information from the test set would leak into training.

In [ ]:
scaler = StandardScaler()

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print("Columns:", X_train.columns.tolist())
print("Train means (should be about 0):", X_train.mean().round(2).tolist())
X_train.head()

## 2.7 Final check before handing over
Make sure there are no missing values and the shapes match.

In [ ]:
print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)
print("Missing in X_train:", X_train.isnull().sum().sum())
print("Missing in X_test: ", X_test.isnull().sum().sum())